# Healthcare FHIR Integration : Évaluation du triage LLM

**Entrées** : 26 tickets de support **fictifs** étiquetés (`ai/support_tickets.json`) et résultats versionnés (`ai/triage_results.json`)  
**LLM** : Llama 3.2 3B en local via Ollama, température 0 (**non relancé** dans ce notebook)  
**Stack** : Python, intervalle de Wilson à 95 % (`ai/evaluate_triage.py`)

---

### Objectif

Un LLM local peut classer des demandes de support d'interopérabilité (erreur de mapping, donnée manquante, problème de connexion...), mais sa réponse ne suffit pas. Ce notebook analyse les résultats déjà mesurés et sépare clairement la classification générative des contrôles déterministes faits par Python.

### Fonctionnement

1. **Catégories fermées** : le LLM choisit dans une liste fixe ; toute réponse hors liste est rejetée par Python.
2. **Trois versions de prompt** comparées sur les mêmes 26 tickets.
3. **Intervalle de Wilson** recalculé indépendamment : sur 26 tickets, un simple pourcentage serait trompeur.

### Limite assumée

Les tickets et le prompt ont été écrits par la même personne sur un très petit jeu : le score est optimiste et illustre une méthode d'évaluation, pas une performance en conditions réelles.

## 1. Chargement des résultats versionnés

Les métriques proviennent d’exécutions déjà mesurées avec Llama 3.2 3B à température 0. On vérifie d’abord leur contrat minimal et leur cohérence avec le nombre de tickets étiquetés.


In [1]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ai.evaluate_triage import wilson_interval

results = json.loads((PROJECT_ROOT / "ai" / "triage_results.json").read_text(encoding="utf-8"))
tickets = json.loads((PROJECT_ROOT / "ai" / "support_tickets.json").read_text(encoding="utf-8"))

assert results["fictional_data"] is True
assert results["total_tickets"] == len(tickets)
print(f"Tickets fictifs : {len(tickets)}")
print(f"Modèle documenté : {results['model']} | température : {results['temperature']}")


Tickets fictifs : 26
Modèle documenté : Llama 3.2 3B | température : 0


> **Contrôle.** Le jeu contient bien 26 tickets fictifs et le nombre concorde avec le fichier de résultats. Cette vérification évite de commenter des scores devenus incompatibles avec le dataset courant.


## 2. Recalcul des métriques

Pour chaque prompt, on recalcule l’accuracy et l’intervalle de Wilson avec la fonction du projet. L’intervalle matérialise l’incertitude d’échantillonnage liée au petit effectif.


In [2]:
rows = []
for run in results["runs"]:
    correct = run["correct"]
    total = results["total_tickets"]
    lower, upper = wilson_interval(correct, total)
    rows.append({
        "prompt": run["prompt_version"],
        "correct": f"{correct}/{total}",
        "accuracy": f"{correct / total:.1%}",
        "wilson_95": f"{lower:.1%}–{upper:.1%}",
    })

rows


[{'prompt': 'Référence',
  'correct': '23/26',
  'accuracy': '88.5%',
  'wilson_95': '71.0%–96.0%'},
 {'prompt': 'Définition plus précise de question_de_format',
  'correct': '22/26',
  'accuracy': '84.6%',
  'wilson_95': '66.5%–93.8%'},
 {'prompt': 'Règle supplémentaire dans le prompt système',
  'correct': '20/26',
  'accuracy': '76.9%',
  'wilson_95': '57.9%–89.0%'}]

> **Résultat.** Le prompt de référence obtient **23/26, soit 88,5 %** avec un IC 95 % de Wilson d’environ **71,0 %–96,0 %**. Les deux modifications testées obtiennent 22/26 puis 20/26 : elles sont donc rejetées plutôt que conservées sur intuition.

> **Observation.** Les intervalles sont larges et se chevauchent. Le classement des prompts décrit ce petit jeu interne ; il ne démontre ni une supériorité stable ni une performance en production.


## 3. Décision d’architecture

Le modèle choisit uniquement une catégorie dans une liste fermée. Python rejette toute catégorie inconnue, conserve les décisions critiques—parsing, validation et sévérité—et permet aux pipelines FHIR/HL7 de fonctionner lorsque Ollama est indisponible.


In [3]:
from ai.ticket_triage import CATEGORIES

print(f"Nombre de catégories autorisées : {len(CATEGORIES)}")
for name, description in CATEGORIES.items():
    print(f"- {name}: {description}")


Nombre de catégories autorisées : 5
- erreur_de_mapping: une valeur est mal convertie ou associée au mauvais champ entre deux formats
- donnee_manquante: un champ ou un enregistrement est vide, absent ou incomplet
- probleme_de_connexion: les échanges avec un système sont coupés, lents ou en erreur (timeout, 500...)
- question_de_format: question sur la structure d'un standard (HL7, FHIR, JSON) ou sur sa documentation
- autre: ne correspond à aucune autre catégorie


> **Décision.** Le LLM reste une aide facultative sur une tâche à faible criticité. Une sortie conforme au JSON n’est pas nécessairement vraie : la validation garantit le contrat syntaxique, pas la qualité sémantique de la catégorie.

## Conclusion générale et prochaines décisions

Les trois notebooks montrent une séparation nette : l’intégration FHIR, le mapping HL7 et la persistance restent déterministes ; le LLM est isolé et évalué sur des données fictives. Les résultats actuels illustrent une méthode d’évaluation, pas une aptitude au déploiement clinique.

**Prochaines décisions :** agrandir le jeu d’évaluation avec des demandes ambiguës et anonymisées ; mesurer la variabilité entre exécutions ; maintenir un jeu de non-régression indépendant de la conception du prompt ; ne traiter aucune donnée réelle avant une revue de sécurité, de conformité et de gouvernance.
